# Notebook 07v5: FastAPI Backend Integration (Production)

**Project:** Game-aware NPC: Player-Based NPC Behavior Generation for Blockchain Gaming  
**Author:** Ramesh Krishnan  
**Date:** February 2026  
**Version:** 5.0 (Production Backend with RL Agent, Modal LLM, Redis Sessions, Action Executor)

---

## Overview

This notebook implements the complete **FastAPI backend** for the Game-aware NPC system, integrating:

1. **Contextual Bandit RL Agent** - Adaptive NPC decision-making (discount, urgency, upsell, loan)
2. **LLM Service** - Modal-hosted fine-tuned model (Qwen v2 ChatML format) with mock fallback
3. **Redis Session Store** - Persistent player sessions across server restarts
4. **Action Executor** - Intent detection and purchase confirmation flow
5. **Stock Manager** - Global inventory with scarcity-driven dynamic pricing
6. **Proactive Agent** - Milestone detection and context-aware commentary
7. **Auto-Registration** - Participant management with counterbalancing

## Architecture

```
Frontend (React) --> FastAPI Backend --> Modal LLM (Qwen2.5 + LoRA)
                         |
                    RL Agent (Contextual Bandit)
                         |
                    Redis (Sessions + RL State)
```

## API Endpoints Summary

| Endpoint | Method | Purpose |
|----------|--------|---------|
| `/health` | GET | Backend health check |
| `/warmup` | POST | Pre-warm Modal container |
| `/diagnose` | GET | Debug Modal connectivity |
| `/survey` | POST | Process pre-game survey, classify archetype |
| `/interact` | POST | Main NPC chat endpoint |
| `/outcome` | POST | Report purchase outcome for RL learning |
| `/session-end` | POST | Handle session cleanup |
| `/proactive` | POST | Context-aware proactive messages |
| `/register` | POST | Auto-register participant |
| `/stock/status` | GET | Current stock levels |
| `/rl/stats` | GET | RL agent statistics |
| `/rl/prewarm` | POST | Pre-warm RL with hypotheses |
| `/admin/export/all` | GET | Export all study data |

---
# Part 1: Setup & Dependencies

In [ ]:
# Cell 1.1: Install Core Dependencies
# ====================================
#
# Core FastAPI stack + async HTTP client + Redis

!pip install -q fastapi==0.109.0 uvicorn[standard]==0.27.0 pydantic==2.5.3 pydantic-settings
!pip install -q python-dotenv httpx==0.26.0 redis==5.0.1 aiofiles nest-asyncio

print("Core dependencies installed")

In [ ]:
# Cell 1.2: Install ML Dependencies (Optional - for LocalLLMService)
# ==================================================================
#
# Only needed if LLM_MODE='local' (running fine-tuned model on GPU).
# Skip if using LLM_MODE='modal' (serverless) or LLM_MODE='mock' (testing).

!pip install -q torch>=2.2.0 transformers>=4.41.0 peft>=0.10.0 bitsandbytes>=0.42.0 accelerate>=0.25.0

print("ML dependencies installed (for LocalLLMService)")

In [ ]:
# Cell 1.3: Verify Installation
# ==============================

import sys
print(f"Python: {sys.version}")

import fastapi
import pydantic
print(f"FastAPI: {fastapi.__version__}")
print(f"Pydantic: {pydantic.__version__}")

# Check PyTorch (optional)
try:
    import torch
    if torch.backends.mps.is_available():
        device = "MPS (Apple Silicon)"
    elif torch.cuda.is_available():
        device = f"CUDA ({torch.cuda.get_device_name(0)})"
    else:
        device = "CPU"
    print(f"PyTorch: {torch.__version__} - Device: {device}")
except ImportError:
    print("PyTorch not installed (LocalLLMService unavailable)")

print()
print("=" * 50)
print("SETUP COMPLETE")
print("=" * 50)

---
# Part 2: Configuration Management

The `Settings` class uses Pydantic's `BaseSettings` for environment-variable-driven configuration.
All secrets (Modal endpoint, Redis URL, admin key) are loaded from `.env` file or environment variables.

**LLM Modes:**
- `modal` -- Production: Uses Modal.com serverless GPU for fine-tuned Qwen2.5 inference
- `mock` -- Testing: Keyword-based mock responses (no GPU required)
- `local` -- Development: Runs model locally (requires GPU with 8GB+ VRAM)

In [ ]:
# Cell 2.1: Settings Configuration
# ==================================

from pydantic_settings import BaseSettings
from typing import Optional, Literal
from functools import lru_cache


class Settings(BaseSettings):
    """
    Application configuration.
    All settings can be overridden via environment variables or .env file.
    """

    # Application
    APP_NAME: str = "Whisper NPC System"
    APP_VERSION: str = "1.0.0"
    DEBUG: bool = True

    # LLM Configuration
    LLM_MODE: Literal["local", "modal", "mock"] = "mock"
    MODEL_PATH: str = "./models/whisper_v3"
    MODAL_ENDPOINT: Optional[str] = None
    MODAL_API_KEY: Optional[str] = None

    # LLM Generation Parameters
    MAX_NEW_TOKENS: int = 100
    TEMPERATURE: float = 0.7
    TOP_P: float = 0.9

    # Performance
    TARGET_LATENCY_MS: int = 500

    # RL Agent
    RL_EPSILON: float = 0.1
    RL_LEARNING_RATE: float = 0.15

    # Redis (for session persistence and RL state backup)
    REDIS_URL: Optional[str] = None

    # Admin
    ADMIN_KEY: str = "admin2026"

    class Config:
        env_file = ".env"
        extra = "ignore"


@lru_cache()
def get_settings() -> Settings:
    return Settings()


# Display current settings
settings = get_settings()
print("Current Settings:")
print(f"  LLM_MODE: {settings.LLM_MODE}")
print(f"  MODEL_PATH: {settings.MODEL_PATH}")
print(f"  DEBUG: {settings.DEBUG}")
print(f"  RL_EPSILON: {settings.RL_EPSILON}")
print(f"  RL_LEARNING_RATE: {settings.RL_LEARNING_RATE}")
print(f"  TARGET_LATENCY_MS: {settings.TARGET_LATENCY_MS}ms")

---
# Part 3: Data Models

## 3.1 Enums

These enums define the discrete action/context spaces used by the RL agent.
They must match the definitions from Notebook 06 (Contextual Bandit) exactly.

**Player Archetypes** (from Notebook 01 K-means clustering):
- 6 clusters identified from behavioral data
- Each archetype has distinct spending patterns and risk tolerance

**RL Action Spaces:**
- Discount: 0%, 10%, 15%, 20%
- Loan: approve variants / deny
- Urgency: low / medium / high / critical
- Upsell: none / hint / premium_hint / scroll / nft variants

In [ ]:
# Cell 3.1: Enums (Matching Notebook 06)
# =======================================
#
# These MUST match the enums from Notebook 06 exactly.
# The RL agent uses these for decision-making.

from enum import Enum
from typing import List, Dict, Optional, Tuple, Any
from pydantic import BaseModel, Field
from datetime import datetime


# === PLAYER ARCHETYPES (from K-means clustering) ===
class PlayerArchetype(str, Enum):
    ACHIEVEMENT_HUNTER = "achievement_hunter"
    ENGAGED_BEGINNER = "engaged_beginner"
    SPENDER = "spender"
    WEEKEND_WARRIOR = "weekend_warrior"
    CASUAL_VETERAN = "casual_veteran"
    TROPHY_HUNTER = "trophy_hunter"


# === RL PLAYER ARCHETYPE (internal enum for RL context) ===
class RLPlayerArchetype(Enum):
    ACHIEVEMENT_HUNTER = "achievement_hunter"
    ENGAGED_BEGINNER = "engaged_beginner"
    SPENDER = "spender"
    WEEKEND_WARRIOR = "weekend_warrior"
    CASUAL_VETERAN = "casual_veteran"
    TROPHY_HUNTER = "trophy_hunter"


# === RL ACTION ENUMS ===
class DiscountAction(Enum):
    NONE = 0
    SMALL = 10
    MEDIUM = 15
    LARGE = 20


class LoanAction(Enum):
    DENY = "deny"
    APPROVE_SMALL = "approve_small"
    APPROVE_MEDIUM = "approve_medium"
    APPROVE_LARGE = "approve_large"


class UrgencyAction(Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"


class UpsellAction(Enum):
    NONE = "none"
    HINT = "hint"
    PREMIUM_HINT = "premium_hint"
    SCROLL = "scroll"
    NFT_COMMON = "nft_common"
    NFT_RARE = "nft_rare"


# === CONTEXT DISCRETIZATION ===
class RelationshipLevel(Enum):
    NEW = "new"         # 0-25
    LOW = "low"         # 26-50
    MEDIUM = "medium"   # 51-75
    HIGH = "high"       # 76-100


class StockLevel(Enum):
    ABUNDANT = "abundant"   # >70%
    NORMAL = "normal"       # 30-70%
    SCARCE = "scarce"       # 10-30%
    LAST_FEW = "last_few"   # <10%


class PlayerRisk(Enum):
    SAFE = "safe"           # 0 curses
    CAUTIOUS = "cautious"   # 1 curse
    DANGER = "danger"       # 2 curses
    CRITICAL = "critical"   # 3 curses


class PointsLevel(Enum):
    BROKE = "broke"     # <100
    LOW = "low"         # 100-250
    MEDIUM = "medium"   # 251-500
    HIGH = "high"       # >500


print(f"Enums defined: {len(PlayerArchetype)} archetypes, {len(DiscountAction)} discount levels")

In [ ]:
# Cell 3.2: Request Models
# =========================
#
# Define what data the frontend sends to the API.
# FastAPI validates incoming JSON against these schemas.

class SurveyAnswers(BaseModel):
    frequency: str
    session_length: str
    motivation: str
    spending: str
    progression: str


class SurveyRequest(BaseModel):
    session_id: str
    answers: SurveyAnswers
    wallet_address: Optional[str] = None
    participant_id: Optional[str] = None


class GameState(BaseModel):
    """Current game state - single source of truth."""
    level: int = Field(..., ge=1, le=7)
    points: int = Field(..., ge=0)
    pol_balance: float = 0.0
    curses: int = Field(0, ge=0, le=4)
    golden_gates_found: int = 0
    has_loan: bool = False
    loan_amount: int = 0
    hints_stock: int = 100
    hints_max: int = 100
    scrolls_stock: int = 50
    scrolls_max: int = 100
    has_merchants_favor: bool = False
    has_shadows_blessing: bool = False
    current_hint_text: Optional[str] = None
    current_solution_text: Optional[Any] = None


class InteractionRequest(BaseModel):
    """Main request for NPC interaction."""
    session_id: str
    player_message: str = Field(..., min_length=1, max_length=500)
    game_state: GameState
    archetype_override: Optional[PlayerArchetype] = None


class OutcomeRequest(BaseModel):
    """Report player's purchase/action outcome for RL learning."""
    session_id: str
    purchased: Optional[bool] = None
    points_spent: Optional[int] = None
    upsell_accepted: Optional[bool] = None
    loan_repaid: Optional[bool] = None
    player_survived_level: Optional[bool] = None
    session_continued: Optional[bool] = None
    curses_after: Optional[int] = None
    final_points: Optional[int] = None
    relationship_score: Optional[int] = None


print("Request models defined: SurveyRequest, InteractionRequest, OutcomeRequest, GameState")

In [ ]:
# Cell 3.3: Response Models
# ==========================
#
# Define what the API returns to the frontend.

class NPCDecisionResponse(BaseModel):
    """RL agent's decision (visible in debug mode)."""
    discount_percent: int
    loan_decision: str
    urgency_level: str
    upsell_recommendation: str
    reasoning: str


class NPCResponse(BaseModel):
    """Main response from Whisper NPC."""
    session_id: str
    dialogue: str
    decision: Optional[NPCDecisionResponse] = None
    relationship_score: int
    relationship_delta: int = 0
    offer_discount_percent: int = 0
    latency_ms: int
    updated_pol_balance: Optional[float] = None
    updated_points: Optional[int] = None
    purchased_item: Optional[str] = None
    pol_amount: Optional[int] = None
    points_gained: Optional[int] = None


class SurveyResponse(BaseModel):
    session_id: str
    archetype: PlayerArchetype
    archetype_description: str
    initial_relationship: int = 50
    assigned_profile_id: str
    starting_points: int
    starting_pol: float


class HealthResponse(BaseModel):
    status: str = "healthy"
    version: str
    llm_mode: str
    llm_loaded: bool
    rl_agent_loaded: bool
    contexts_seen: int
    redis_sessions_connected: bool = False


class SessionResponse(BaseModel):
    session_id: str
    archetype: PlayerArchetype
    relationship_score: int
    interaction_count: int
    total_spent_points: int


print("Response models defined: NPCResponse, SurveyResponse, HealthResponse, SessionResponse")

---
# Part 4: RL Agent (Contextual Bandit)

The RL agent uses **four independent contextual bandits** (one per action dimension):
- **Discount Bandit** -- Decides discount percentage (0/10/15/20%)
- **Urgency Bandit** -- Decides urgency level (low/medium/high/critical)
- **Upsell Bandit** -- Decides which item to recommend
- **Loan Bandit** -- Decides loan approval/denial

Each bandit uses **Epsilon-Greedy + UCB** (Upper Confidence Bound) exploration.

**Learning Flow:**
1. `/interact` -- Agent observes context, selects actions
2. `/outcome` -- Frontend reports result (purchased? survived? quit?)
3. Agent updates Q-values for the selected actions

**Pre-warming:**
Research hypotheses (from Notebook 05) seed initial Q-values so the agent makes
informed decisions from the first interaction rather than starting from uniform 0.5.

In [ ]:
# Cell 4.1: GameContext (from Notebook 06)
# =========================================
#
# Converts continuous game values to discrete categories for RL.
# Example: points=350 -> PointsLevel.MEDIUM, relationship=72 -> RelationshipLevel.MEDIUM

from dataclasses import dataclass, field
import numpy as np
import random
import json
from collections import defaultdict


@dataclass(frozen=True)
class GameContext:
    """
    Discretized context for RL decision-making.
    frozen=True makes it hashable (can be used as dictionary key).
    """
    archetype: RLPlayerArchetype
    relationship: RelationshipLevel
    stock: StockLevel
    risk: PlayerRisk
    points: PointsLevel
    has_debt: bool = False
    interaction_type: str = "purchase"

    def to_key(self) -> str:
        return f"{self.archetype.value}|{self.relationship.value}|{self.stock.value}|{self.risk.value}|{self.points.value}|{self.has_debt}|{self.interaction_type}"

    def to_learning_key(self) -> str:
        """Simplified context for bandit learning (24 combos vs 3,072)."""
        return f"{self.archetype.value}|{self.risk.value}"

    @classmethod
    def from_raw(cls, archetype: str, relationship_score: int, stock_percent: float,
                 curses: int, points: int, has_debt: bool = False,
                 interaction_type: str = "purchase") -> "GameContext":
        # Relationship
        if relationship_score <= 25:
            rel = RelationshipLevel.NEW
        elif relationship_score <= 50:
            rel = RelationshipLevel.LOW
        elif relationship_score <= 75:
            rel = RelationshipLevel.MEDIUM
        else:
            rel = RelationshipLevel.HIGH

        # Stock
        if stock_percent > 70:
            stk = StockLevel.ABUNDANT
        elif stock_percent > 30:
            stk = StockLevel.NORMAL
        elif stock_percent > 10:
            stk = StockLevel.SCARCE
        else:
            stk = StockLevel.LAST_FEW

        # Risk
        risk_map = {0: PlayerRisk.SAFE, 1: PlayerRisk.CAUTIOUS, 2: PlayerRisk.DANGER, 3: PlayerRisk.CRITICAL}
        rsk = risk_map.get(curses, PlayerRisk.CRITICAL)

        # Points
        if points < 100:
            pts = PointsLevel.BROKE
        elif points <= 250:
            pts = PointsLevel.LOW
        elif points <= 500:
            pts = PointsLevel.MEDIUM
        else:
            pts = PointsLevel.HIGH

        # Archetype
        arch_map = {
            "achievement_hunter": RLPlayerArchetype.ACHIEVEMENT_HUNTER,
            "engaged_beginner": RLPlayerArchetype.ENGAGED_BEGINNER,
            "spender": RLPlayerArchetype.SPENDER,
            "weekend_warrior": RLPlayerArchetype.WEEKEND_WARRIOR,
            "casual_veteran": RLPlayerArchetype.CASUAL_VETERAN,
            "trophy_hunter": RLPlayerArchetype.TROPHY_HUNTER,
        }
        arch = arch_map.get(archetype.lower().replace(" ", "_"), RLPlayerArchetype.ENGAGED_BEGINNER)

        return cls(archetype=arch, relationship=rel, stock=stk, risk=rsk,
                   points=pts, has_debt=has_debt, interaction_type=interaction_type)


print("GameContext defined with from_raw() factory method")

In [ ]:
# Cell 4.2: NPCDecision
# ======================
#
# Output from the RL agent -- contains all action decisions.

@dataclass
class NPCDecision:
    """Output from the RL agent."""
    discount: DiscountAction
    loan: LoanAction
    urgency: UrgencyAction
    upsell: UpsellAction
    reasoning: str = ""

    def to_prompt_context(self) -> str:
        """Format decision for LLM prompt (training-aligned format)."""
        if self.upsell != UpsellAction.NONE:
            action = "upsell"
        elif self.discount != DiscountAction.NONE:
            action = "discount_offer"
        else:
            action = "standard_offer"

        return f"Action: {action} | Discount: {self.discount.value}% | Urgency: {self.urgency.value}"

    def to_dict(self) -> dict:
        return {
            "discount_percent": self.discount.value,
            "loan_decision": self.loan.value,
            "urgency_level": self.urgency.value,
            "upsell_recommendation": self.upsell.value,
            "reasoning": self.reasoning
        }


print("NPCDecision defined with to_prompt_context() and to_dict()")

In [ ]:
# Cell 4.3: Rule-Based Policy (Cold Start)
# ==========================================
#
# Heuristic policy used when the bandit has never seen a context before.
# Encodes domain knowledge about player archetypes and game situations.

class RuleBasedPolicy:
    """Heuristic policy for cold start."""

    def decide(self, context: GameContext) -> NPCDecision:
        discount = DiscountAction.NONE
        loan = LoanAction.DENY
        urgency = UrgencyAction.LOW
        upsell = UpsellAction.NONE
        reasoning_parts = []

        # DISCOUNT RULES
        if context.stock in [StockLevel.SCARCE, StockLevel.LAST_FEW]:
            discount = DiscountAction.NONE
            reasoning_parts.append("No discount: scarce stock")
        elif context.relationship == RelationshipLevel.HIGH and context.stock == StockLevel.ABUNDANT:
            discount = DiscountAction.MEDIUM
            reasoning_parts.append("15% discount: loyal + abundant")
        elif context.archetype == RLPlayerArchetype.SPENDER:
            discount = DiscountAction.SMALL
            reasoning_parts.append("10% discount: spender")

        # LOAN RULES
        if context.has_debt:
            loan = LoanAction.DENY
            reasoning_parts.append("Loan denied: existing debt")
        elif context.interaction_type == "loan":
            if context.relationship == RelationshipLevel.HIGH:
                loan = LoanAction.APPROVE_LARGE
                reasoning_parts.append("Large loan: high relationship")
            elif context.relationship == RelationshipLevel.MEDIUM:
                loan = LoanAction.APPROVE_MEDIUM
                reasoning_parts.append("Medium loan: medium relationship")

        # URGENCY RULES
        if context.stock == StockLevel.LAST_FEW:
            urgency = UrgencyAction.CRITICAL
            reasoning_parts.append("Critical: last few items")
        elif context.stock == StockLevel.SCARCE:
            urgency = UrgencyAction.HIGH
            reasoning_parts.append("High urgency: scarce")
        elif context.risk == PlayerRisk.CRITICAL:
            urgency = UrgencyAction.CRITICAL
            reasoning_parts.append("Critical: 3 curses")
        elif context.risk == PlayerRisk.DANGER:
            urgency = UrgencyAction.HIGH
            reasoning_parts.append("High urgency: 2 curses")

        # UPSELL RULES
        if context.archetype == RLPlayerArchetype.ENGAGED_BEGINNER:
            upsell = UpsellAction.SCROLL if context.points == PointsLevel.HIGH else UpsellAction.HINT
            reasoning_parts.append("Upsell for beginner")
        elif context.archetype == RLPlayerArchetype.SPENDER:
            upsell = UpsellAction.NFT_RARE if context.relationship == RelationshipLevel.HIGH else UpsellAction.NFT_COMMON
            reasoning_parts.append("NFT upsell for spender")
        elif context.risk in [PlayerRisk.DANGER, PlayerRisk.CRITICAL]:
            upsell = UpsellAction.SCROLL
            reasoning_parts.append("Scroll for curse protection")

        return NPCDecision(
            discount=discount, loan=loan, urgency=urgency, upsell=upsell,
            reasoning=" | ".join(reasoning_parts) if reasoning_parts else "Default policy"
        )


print("RuleBasedPolicy defined")

In [ ]:
# Cell 4.4: Contextual Bandit
# ============================
#
# Epsilon-Greedy + UCB bandit for one action dimension.

class ContextualBandit:
    """Epsilon-Greedy + UCB bandit for one action type."""

    def __init__(self, actions: list, epsilon: float = 0.1, ucb_c: float = 1.0, learning_rate: float = 0.1):
        self.actions = actions
        self.epsilon = epsilon
        self.ucb_c = ucb_c
        self.learning_rate = learning_rate
        self.q_values: Dict[str, Dict] = defaultdict(lambda: {a: 0.5 for a in self.actions})
        self.action_counts: Dict[str, Dict] = defaultdict(lambda: {a: 0 for a in self.actions})
        self.context_counts: Dict[str, int] = defaultdict(int)
        self.history: List[Dict] = []

    def _get_ucb_values(self, context_key: str) -> Dict:
        n_total = max(1, self.context_counts[context_key])
        ucb_values = {}
        for action in self.actions:
            q = self.q_values[context_key][action]
            n_action = max(1, self.action_counts[context_key][action])
            ucb_bonus = self.ucb_c * np.sqrt(np.log(n_total) / n_action)
            ucb_values[action] = q + ucb_bonus
        return ucb_values

    def select_action(self, context: GameContext, explore: bool = True):
        context_key = context.to_learning_key()
        if explore and random.random() < self.epsilon:
            return random.choice(self.actions)
        ucb_values = self._get_ucb_values(context_key)
        return max(ucb_values, key=ucb_values.get)

    def update(self, context: GameContext, action, reward: float):
        context_key = context.to_learning_key()
        self.context_counts[context_key] += 1
        self.action_counts[context_key][action] += 1
        old_q = self.q_values[context_key][action]
        new_q = old_q + self.learning_rate * (reward - old_q)
        self.q_values[context_key][action] = new_q
        self.history.append({"context_key": context_key, "action": str(action), "reward": reward})


print("ContextualBandit defined (Epsilon-Greedy + UCB)")

In [ ]:
# Cell 4.5: Pre-warming Hypotheses
# ==================================
#
# Research-based success rate estimates for each (archetype, risk) combination.
# Used to seed Q-values so the agent starts with informed priors.

PREWARM_HYPOTHESES = {
    # Trophy Hunter: Responds to scarcity, doesn't need discounts
    ("trophy_hunter", "safe"): {"discount": {0: 0.7, 10: 0.6, 15: 0.5, 20: 0.4}, "urgency": {"low": 0.5, "medium": 0.6, "high": 0.7, "critical": 0.6}},
    ("trophy_hunter", "cautious"): {"discount": {0: 0.65, 10: 0.6, 15: 0.55, 20: 0.45}, "urgency": {"low": 0.4, "medium": 0.6, "high": 0.75, "critical": 0.7}},
    ("trophy_hunter", "danger"): {"discount": {0: 0.6, 10: 0.6, 15: 0.6, 20: 0.5}, "urgency": {"low": 0.3, "medium": 0.5, "high": 0.8, "critical": 0.85}},
    ("trophy_hunter", "critical"): {"discount": {0: 0.5, 10: 0.55, 15: 0.6, 20: 0.55}, "urgency": {"low": 0.2, "medium": 0.4, "high": 0.7, "critical": 0.9}},

    # Engaged Beginner: Foot-in-door, responds to small discounts
    ("engaged_beginner", "safe"): {"discount": {0: 0.4, 10: 0.6, 15: 0.65, 20: 0.5}, "urgency": {"low": 0.6, "medium": 0.5, "high": 0.4, "critical": 0.3}},
    ("engaged_beginner", "cautious"): {"discount": {0: 0.35, 10: 0.55, 15: 0.7, 20: 0.6}, "urgency": {"low": 0.5, "medium": 0.6, "high": 0.55, "critical": 0.4}},
    ("engaged_beginner", "danger"): {"discount": {0: 0.3, 10: 0.5, 15: 0.7, 20: 0.7}, "urgency": {"low": 0.4, "medium": 0.6, "high": 0.7, "critical": 0.6}},
    ("engaged_beginner", "critical"): {"discount": {0: 0.25, 10: 0.45, 15: 0.65, 20: 0.75}, "urgency": {"low": 0.3, "medium": 0.5, "high": 0.75, "critical": 0.8}},

    # Achievement Hunter: Completion framing, moderate discount sensitivity
    ("achievement_hunter", "safe"): {"discount": {0: 0.5, 10: 0.6, 15: 0.6, 20: 0.5}, "urgency": {"low": 0.5, "medium": 0.6, "high": 0.65, "critical": 0.55}},
    ("achievement_hunter", "cautious"): {"discount": {0: 0.45, 10: 0.6, 15: 0.65, 20: 0.55}, "urgency": {"low": 0.4, "medium": 0.6, "high": 0.7, "critical": 0.65}},
    ("achievement_hunter", "danger"): {"discount": {0: 0.4, 10: 0.55, 15: 0.7, 20: 0.65}, "urgency": {"low": 0.35, "medium": 0.55, "high": 0.75, "critical": 0.8}},
    ("achievement_hunter", "critical"): {"discount": {0: 0.35, 10: 0.5, 15: 0.65, 20: 0.7}, "urgency": {"low": 0.3, "medium": 0.5, "high": 0.7, "critical": 0.85}},

    # Casual Veteran: Value/efficiency focused
    ("casual_veteran", "safe"): {"discount": {0: 0.3, 10: 0.5, 15: 0.6, 20: 0.55}, "urgency": {"low": 0.6, "medium": 0.55, "high": 0.45, "critical": 0.35}},
    ("casual_veteran", "cautious"): {"discount": {0: 0.35, 10: 0.55, 15: 0.65, 20: 0.6}, "urgency": {"low": 0.5, "medium": 0.6, "high": 0.55, "critical": 0.45}},
    ("casual_veteran", "danger"): {"discount": {0: 0.4, 10: 0.55, 15: 0.7, 20: 0.65}, "urgency": {"low": 0.4, "medium": 0.6, "high": 0.65, "critical": 0.6}},
    ("casual_veteran", "critical"): {"discount": {0: 0.45, 10: 0.55, 15: 0.65, 20: 0.7}, "urgency": {"low": 0.35, "medium": 0.55, "high": 0.7, "critical": 0.75}},

    # Weekend Warrior: Time-saving, convenience focused
    ("weekend_warrior", "safe"): {"discount": {0: 0.4, 10: 0.5, 15: 0.55, 20: 0.5}, "urgency": {"low": 0.5, "medium": 0.55, "high": 0.5, "critical": 0.4}},
    ("weekend_warrior", "cautious"): {"discount": {0: 0.4, 10: 0.55, 15: 0.6, 20: 0.55}, "urgency": {"low": 0.45, "medium": 0.6, "high": 0.6, "critical": 0.5}},
    ("weekend_warrior", "danger"): {"discount": {0: 0.45, 10: 0.55, 15: 0.65, 20: 0.6}, "urgency": {"low": 0.4, "medium": 0.6, "high": 0.7, "critical": 0.65}},
    ("weekend_warrior", "critical"): {"discount": {0: 0.45, 10: 0.55, 15: 0.6, 20: 0.65}, "urgency": {"low": 0.35, "medium": 0.55, "high": 0.7, "critical": 0.8}},

    # Spender: Already buys, optimize for upsell
    ("spender", "safe"): {"discount": {0: 0.7, 10: 0.65, 15: 0.6, 20: 0.5}, "urgency": {"low": 0.6, "medium": 0.65, "high": 0.6, "critical": 0.5}},
    ("spender", "cautious"): {"discount": {0: 0.65, 10: 0.65, 15: 0.6, 20: 0.55}, "urgency": {"low": 0.55, "medium": 0.65, "high": 0.7, "critical": 0.6}},
    ("spender", "danger"): {"discount": {0: 0.6, 10: 0.6, 15: 0.65, 20: 0.6}, "urgency": {"low": 0.5, "medium": 0.6, "high": 0.75, "critical": 0.7}},
    ("spender", "critical"): {"discount": {0: 0.55, 10: 0.6, 15: 0.65, 20: 0.65}, "urgency": {"low": 0.45, "medium": 0.6, "high": 0.7, "critical": 0.8}},
}

# Upsell success rates by archetype
UPSELL_HYPOTHESES = {
    "trophy_hunter": {"none": 0.4, "hint": 0.45, "premium_hint": 0.5, "scroll": 0.55, "nft_common": 0.65, "nft_rare": 0.7},
    "engaged_beginner": {"none": 0.5, "hint": 0.65, "premium_hint": 0.6, "scroll": 0.55, "nft_common": 0.4, "nft_rare": 0.3},
    "achievement_hunter": {"none": 0.45, "hint": 0.55, "premium_hint": 0.55, "scroll": 0.6, "nft_common": 0.55, "nft_rare": 0.5},
    "casual_veteran": {"none": 0.5, "hint": 0.5, "premium_hint": 0.5, "scroll": 0.55, "nft_common": 0.5, "nft_rare": 0.45},
    "weekend_warrior": {"none": 0.55, "hint": 0.5, "premium_hint": 0.5, "scroll": 0.5, "nft_common": 0.45, "nft_rare": 0.4},
    "spender": {"none": 0.3, "hint": 0.5, "premium_hint": 0.55, "scroll": 0.65, "nft_common": 0.75, "nft_rare": 0.8},
}

print(f"Pre-warming hypotheses defined: {len(PREWARM_HYPOTHESES)} contexts, {len(UPSELL_HYPOTHESES)} archetype upsell profiles")

In [ ]:
# Cell 4.6: WhisperDecisionAgent
# ================================
#
# Complete RL decision agent combining:
# - Rule-based cold start policy
# - Four contextual bandits (discount, loan, urgency, upsell)
# - Pre-warming from research hypotheses
# - Redis persistence for Q-values

class WhisperDecisionAgent:
    """Complete RL decision agent for Whisper NPC."""

    def __init__(self, epsilon: float = 0.1, learning_rate: float = 0.1, use_rules_for_cold_start: bool = True):
        self.epsilon = epsilon
        self.learning_rate = learning_rate
        self.rule_policy = RuleBasedPolicy()
        self.use_rules_for_cold_start = use_rules_for_cold_start

        self.discount_bandit = ContextualBandit(list(DiscountAction), epsilon, learning_rate=learning_rate)
        self.loan_bandit = ContextualBandit(list(LoanAction), epsilon, learning_rate=learning_rate)
        self.urgency_bandit = ContextualBandit(list(UrgencyAction), epsilon, learning_rate=learning_rate)
        self.upsell_bandit = ContextualBandit(list(UpsellAction), epsilon, learning_rate=learning_rate)
        self.seen_contexts: set = set()

    def decide(self, context: GameContext, explore: bool = True) -> NPCDecision:
        context_key = context.to_key()
        learning_key = context.to_learning_key()

        # Cold start: use rules if neither key has been seen
        if self.use_rules_for_cold_start and context_key not in self.seen_contexts and learning_key not in self.seen_contexts:
            self.seen_contexts.add(context_key)
            return self.rule_policy.decide(context)
        self.seen_contexts.add(context_key)

        # Bandit selection
        discount = self.discount_bandit.select_action(context, explore)
        loan = self.loan_bandit.select_action(context, explore)
        urgency = self.urgency_bandit.select_action(context, explore)
        upsell = self.upsell_bandit.select_action(context, explore)

        if context.has_debt:
            loan = LoanAction.DENY

        return NPCDecision(discount=discount, loan=loan, urgency=urgency, upsell=upsell, reasoning="Bandit-selected")

    def update_from_outcome(self, context: GameContext, decision: NPCDecision, outcome: Dict[str, float]):
        """Update bandits from observed outcomes."""
        if 'purchased' in outcome:
            self.discount_bandit.update(context, decision.discount, outcome['purchased'])
            self.urgency_bandit.update(context, decision.urgency, outcome['purchased'])
        if 'loan_repaid' in outcome:
            self.loan_bandit.update(context, decision.loan, outcome['loan_repaid'])
        if 'upsell_accepted' in outcome:
            self.upsell_bandit.update(context, decision.upsell, outcome['upsell_accepted'])
        if 'player_survived_level' in outcome:
            bonus = 0.2 if outcome['player_survived_level'] > 0 else -0.1
            for bandit, action in [(self.discount_bandit, decision.discount), (self.urgency_bandit, decision.urgency),
                                   (self.loan_bandit, decision.loan), (self.upsell_bandit, decision.upsell)]:
                bandit.update(context, action, 0.5 + bonus)
        if 'player_quit' in outcome:
            penalty = -0.3
            for bandit, action in [(self.discount_bandit, decision.discount), (self.urgency_bandit, decision.urgency),
                                   (self.loan_bandit, decision.loan), (self.upsell_bandit, decision.upsell)]:
                bandit.update(context, action, 0.5 + penalty)
        if 'relationship_delta' in outcome:
            rel_reward = min(1.0, max(0.0, 0.5 + outcome['relationship_delta'] * 0.1))
            self.discount_bandit.update(context, decision.discount, rel_reward)
            self.urgency_bandit.update(context, decision.urgency, rel_reward)

    def prewarm(self, n_iterations: int = 200) -> dict:
        """Pre-warm RL agent with simulated interactions from research hypotheses."""
        archetypes = ["trophy_hunter", "engaged_beginner", "achievement_hunter",
                      "casual_veteran", "weekend_warrior", "spender"]
        risk_levels = ["safe", "cautious", "danger", "critical"]
        stats = {"iterations": n_iterations, "discount_updates": 0, "urgency_updates": 0,
                 "upsell_updates": 0, "contexts_seeded": set()}

        for i in range(n_iterations):
            archetype = random.choice(archetypes)
            risk = random.choice(risk_levels)
            context = GameContext(archetype=RLPlayerArchetype(archetype), relationship=RelationshipLevel.NEW,
                                 stock=StockLevel.NORMAL, risk=PlayerRisk(risk), points=PointsLevel.MEDIUM)
            learning_key = context.to_learning_key()
            stats["contexts_seeded"].add(learning_key)
            hypothesis = PREWARM_HYPOTHESES.get((archetype, risk), {})

            if "discount" in hypothesis:
                action = self.discount_bandit.select_action(context)
                success_rate = hypothesis["discount"].get(action.value, 0.5)
                self.discount_bandit.update(context, action, 1.0 if random.random() < success_rate else 0.0)
                stats["discount_updates"] += 1
            if "urgency" in hypothesis:
                action = self.urgency_bandit.select_action(context)
                success_rate = hypothesis["urgency"].get(action.value, 0.5)
                self.urgency_bandit.update(context, action, 1.0 if random.random() < success_rate else 0.0)
                stats["urgency_updates"] += 1
            if i % 5 == 0:
                upsell_hyp = UPSELL_HYPOTHESES.get(archetype, {})
                action = self.upsell_bandit.select_action(context)
                success_rate = upsell_hyp.get(action.value, 0.5)
                self.upsell_bandit.update(context, action, 1.0 if random.random() < success_rate else 0.0)
                stats["upsell_updates"] += 1
            self.seen_contexts.add(learning_key)

        stats["contexts_seeded"] = list(stats["contexts_seeded"])
        stats["total_contexts"] = len(stats["contexts_seeded"])
        print(f"Pre-warmed RL agent: {n_iterations} iterations across {stats['total_contexts']} contexts")
        return stats

    def get_stats(self) -> Dict:
        return {
            "contexts_seen": len(self.seen_contexts),
            "discount_history": len(self.discount_bandit.history),
            "loan_history": len(self.loan_bandit.history),
            "urgency_history": len(self.urgency_bandit.history),
            "upsell_history": len(self.upsell_bandit.history),
        }


print("WhisperDecisionAgent defined")

In [ ]:
# Cell 4.7: Test RL Agent Integration
# =====================================
#
# Verify the RL agent works before building the API.

# Create agent
agent = WhisperDecisionAgent(epsilon=0.1, learning_rate=0.15)

# Test with mock game context
test_context = GameContext.from_raw(
    archetype="spender",
    relationship_score=60,
    stock_percent=45.0,
    curses=2,
    points=300,
    has_debt=False
)

# Get decision (cold start = rule-based)
decision = agent.decide(test_context, explore=True)

print("Test RL Decision:")
print(f"  Context: archetype={test_context.archetype.value}, risk={test_context.risk.value}")
print(f"  Discount: {decision.discount.value}%")
print(f"  Urgency: {decision.urgency.value}")
print(f"  Upsell: {decision.upsell.value}")
print(f"  Loan: {decision.loan.value}")
print(f"  Reasoning: {decision.reasoning}")

# Pre-warm with hypotheses
stats = agent.prewarm(n_iterations=200)

# Verify second decision uses bandit (not rules)
decision2 = agent.decide(test_context, explore=True)
print(f"\nPost-prewarm decision: {decision2.reasoning}")
print(f"Agent stats: {agent.get_stats()}")

---
# Part 5: LLM Service

The LLM generates Whisper's dialogue based on:
1. **System prompt** -- Character personality, game rules, response constraints
2. **Game state** -- Level, points, curses, stock levels
3. **RL decision** -- Discount, urgency, upsell (controls strategy)
4. **Conversation history** -- Last 3 turns for continuity

**LLM Format:** ChatML (Qwen v2) -- `<|im_start|>system ... <|im_end|>`

**Three Backends:**
- `ModalLLMService` -- Production (serverless GPU on Modal.com)
- `MockLLMService` -- Testing (keyword-based, no GPU needed)
- `LocalLLMService` -- Development (local GPU with LoRA model)

In [ ]:
# Cell 5.1: System Prompt
# ========================
#
# Defines Whisper's personality, game rules, and response constraints.
# The RL decision (passed in the user context) overrides Whisper's strategy.

WHISPER_SYSTEM_PROMPT = '''You are Whisper, a chill merchant in "Origins of Lume: Gate of Whispers."

## WHO YOU ARE
- Mysterious but approachable merchant at the Gate of Whispers
- Speaks casual fantasy - like a millennial who runs a magic shop
- Honest, strategic, adapts to players

## PERSONALITY & BACKGROUND
- You appeared at the gates three winters ago - no one knows where from
- You've seen countless seekers come and go, some victorious, many not
- You're not cruel, but you're practical - this is business, after all
- You genuinely want players to succeed, but you won't give handouts
- When pressed about your past, you deflect with mystery or humor

## LANGUAGE STYLE
- Short sentences, contractions always ("don't", "can't", "you're")
- Casual: "bet", "gotchu", "for sure", "ngl", "lowkey"
- Light mystical flavor when it fits ("shadows", "gate")
- AVOID: "traveler", "shall", "illuminate your path", "your coffers"

## ACTION COMPLIANCE (CRITICAL)
The [RL DECISION] Action directive MUST be honored exactly:
- empathy_first: Lead with emotional understanding FIRST, validate feelings
- deny_loan: Politely decline credit requests with reason
- collect_debt: Request loan repayment before any new transactions
- identity_answer: Answer questions about yourself - NO sales
- none: Standard conversation - NO sales pitch at all
- standard_offer: Normal pricing, balanced approach
- upsell: Suggest higher-value items appropriately
- push_scroll: Strongly recommend scroll (for curse protection)
- offer_discount: Apply discount with clear reason
- scarcity: Mention low stock (only when true in context)
- deescalate: Calm panicking/angry players first
- teach: Explain game rules without sales

## LOAN RULES
- IF [LOAN STATUS: has_debt] -> Address debt BEFORE any new sales
- IF [LOAN STATUS: overdue] -> Request repayment, deny new loans
- IF Action: deny_loan -> Decline politely with reason, suggest alternatives
- IF Action: collect_debt -> Firmly but kindly request payment

## URGENCY RULES
- IF Urgency: critical -> Use urgent language ("now", "before it's too late")
- IF Urgency: high -> Moderate urgency ("might want to act soon")
- IF Urgency: medium -> Balanced tone
- IF Urgency: low -> Relaxed, no pressure

## STATE-AWARE RULES
- IF curses >= 3 -> Prioritize safety (scroll) over profit
- IF points < item_price -> Say "can't afford", offer alternatives
- IF POL < 15 -> Don't push NFTs
- IF discount = 0% -> NEVER mention discounts

## NUMERIC AUTHORITY
- If [EFFECTIVE PRICES] exists, use those numbers exactly.
- Never invent or estimate prices, points, stock, debt, discounts, or remaining balance.

## RESPONSE RULES
1. IF player expresses emotion -> Acknowledge emotion FIRST
2. IF player asks identity/lore -> Answer directly, NO selling
3. IF player asks price -> State price from [EFFECTIVE PRICES]
4. Keep responses 15-40 words
5. HONOR [RL DECISION] exactly
6. NEVER invent prices or discounts
7. NEVER mention player archetype names
8. For real-world trivia or off-topic questions: deflect in character, return to the Gate'''


print(f"System prompt defined ({len(WHISPER_SYSTEM_PROMPT)} chars)")

In [ ]:
# Cell 5.2: LLM Service Interface & Implementations
# ===================================================

import time
import asyncio
import httpx
from abc import ABC, abstractmethod


class LLMService(ABC):
    @abstractmethod
    async def generate(self, prompt: str) -> Tuple[str, int]:
        pass

    @abstractmethod
    async def health_check(self) -> bool:
        pass


class MockLLMService(LLMService):
    """Context-aware mock responses based on RL decision."""

    async def generate(self, prompt: str) -> Tuple[str, int]:
        await asyncio.sleep(0.1)
        if "Critical" in prompt or "critical" in prompt:
            response = "Yo, your curses are stacking up bad. Grab a Gate Scroll for 250 points before it's too late."
        elif "Discount: 15%" in prompt or "Discount: 10%" in prompt:
            response = "Got a deal for you - special price today. The shadows favor those who act wisely."
        elif "Upsell: Recommend scroll" in prompt:
            response = "Haunted gates are no joke. A Gate Scroll for 250 points removes that risk entirely."
        elif "Upsell: Recommend hint" in prompt:
            response = "Stuck on the puzzle? A hint for 150 points might help you out."
        elif "upsell" in prompt.lower() and "nft" in prompt.lower():
            response = "The Merchant's Favor gives permanent discounts. 15 POL for lasting fortune."
        else:
            response = "Hey there, seeker. What can Whisper help you with today?"
        return response, 100

    async def health_check(self) -> bool:
        return True


class ModalLLMService(LLMService):
    """Remote LLM via Modal.com serverless GPU."""

    def __init__(self, endpoint: str, api_key: Optional[str] = None):
        self.endpoint = endpoint
        self.api_key = api_key
        self.client = httpx.AsyncClient(timeout=60.0)
        self.max_retries = 2

    async def generate(self, prompt: str) -> Tuple[str, int]:
        start = time.time()
        headers = {"Authorization": f"Bearer {self.api_key}"} if self.api_key else {}

        last_error = None
        for attempt in range(self.max_retries + 1):
            try:
                response = await self.client.post(
                    self.endpoint,
                    json={"prompt": prompt, "max_new_tokens": 150, "temperature": 0.7},
                    headers=headers
                )
                response.raise_for_status()
                text = response.json().get("response", "")
                latency = int((time.time() - start) * 1000)
                if attempt > 0:
                    print(f"Modal succeeded on retry {attempt}, latency: {latency}ms")
                return text, latency
            except httpx.TimeoutException:
                last_error = f"Timeout (attempt {attempt+1}/{self.max_retries+1})"
                print(f"Modal timeout: {last_error}")
            except httpx.ConnectError as e:
                last_error = f"Connection failed: {e}"
                print(f"Modal connect error: {last_error}")
            except httpx.HTTPStatusError as e:
                last_error = f"HTTP {e.response.status_code}"
                print(f"Modal HTTP error: {last_error}")
                break
            except Exception as e:
                last_error = str(e)
                print(f"Modal error: {last_error}")
            if attempt < self.max_retries:
                await asyncio.sleep(2 ** attempt)

        print(f"Modal FAILED after {self.max_retries+1} attempts: {last_error}")
        return "The shadows grow thick... Ask again, traveler.", int((time.time() - start) * 1000)

    async def health_check(self) -> bool:
        try:
            health_url = self.endpoint.replace("-generate.", "-health.")
            r = await self.client.get(health_url, timeout=10)
            return r.status_code == 200
        except Exception as e:
            print(f"Modal health check failed: {e}")
            return False


print("LLM Services defined: MockLLMService, ModalLLMService")

In [ ]:
# Cell 5.3: Prompt Builder (Qwen v2 ChatML Format)
# ==================================================
#
# Builds the complete LLM prompt including game state, RL decision,
# conversation history, and effective prices (with NFT discounts).

def build_prompt(player_message: str, game_state: dict, decision: NPCDecision,
                 archetype: str, relationship: str, conversation_history: list = None) -> str:
    """Build LLM prompt with RL decision included (Qwen v2 ChatML format)."""
    # Compute effective prices: NFT discount from game state, RL discount from agent
    nft_discount = 0
    if game_state.get('has_shadows_blessing', False):
        nft_discount = 30
    elif game_state.get('has_merchants_favor', False):
        nft_discount = 15

    effective_discount = max(nft_discount, decision.discount.value)
    multiplier = 1 - effective_discount / 100
    hint_price = int(150 * multiplier)
    solution_price = int(300 * multiplier)
    scroll_price = int(250 * multiplier)

    # Game state fields
    level = game_state.get('level', 1)
    golden_gates = game_state.get('golden_gates_found', 0)
    hints_stock = game_state.get('hints_stock', 100)
    hints_max = game_state.get('hints_max', 100)
    scrolls_stock = game_state.get('scrolls_stock', 50)
    scrolls_max = game_state.get('scrolls_max', 100)
    hints_pct = int(hints_stock / max(1, hints_max) * 100)
    scrolls_pct = int(scrolls_stock / max(1, scrolls_max) * 100)
    points = game_state.get('points', 0)
    pol = game_state.get('pol_balance', 10)
    curses = game_state.get('curses', 0)

    # Build conversation history block
    history_block = ""
    if conversation_history and len(conversation_history) > 0:
        history_lines = []
        for turn in conversation_history[-3:]:
            if 'player' in turn and 'whisper' in turn:
                history_lines.append(f"Player: {turn['player']}")
                history_lines.append(f"Whisper: {turn['whisper']}")
        if history_lines:
            history_block = "\n[CONVERSATION HISTORY]\n" + "\n".join(history_lines) + "\n"

    user_context = f"""[GAME STATE]
Level: {level}/7 | Golden Gates: {golden_gates}/5
Stock: Hints {hints_pct}% | Scrolls {scrolls_pct}%

[PLAYER STATE]
Points: {points} | POL: {pol} | Curses: {curses}/4

[EFFECTIVE PRICES]
Hint: {hint_price} pts | Solution: {solution_price} pts | Scroll: {scroll_price} pts
Common NFT: 15 POL | Rare NFT: 25 POL
{history_block}
[RL DECISION]
{decision.to_prompt_context()}

Player: {player_message}"""

    return f"""<|im_start|>system
{WHISPER_SYSTEM_PROMPT}<|im_end|>
<|im_start|>user
{user_context}<|im_end|>
<|im_start|>assistant
"""


print("Prompt builder defined (Qwen v2 ChatML format)")

In [ ]:
# Cell 5.4: LLM Response Cleaner
# ================================
#
# Removes training artifacts from LLM output (ChatML tokens, phantom numbers, etc.)

import re

def clean_llm_response(response: str) -> str:
    """Remove training artifacts from LLM output (Qwen v2 aligned)."""
    cleaned = response

    # ChatML artifact removal
    cleaned = re.sub(r'<\|im_start\|>(system|user|assistant)?', '', cleaned)
    cleaned = re.sub(r'<\|im_end\|>', '', cleaned)
    cleaned = re.sub(r'<\|endoftext\|>', '', cleaned)

    # Phantom number fix (model sometimes outputs "2250" instead of "250")
    cleaned = re.sub(r'\b2(150|250|300)\b', r'\1', cleaned)

    # Remove internal markers
    cut_markers = [
        r'\[decision\]:.*', r'\[DECISION\]:.*', r'\[GAME STATE\].*',
        r'\[PLAYER STATE\].*', r'\[CONTEXT\].*', r'\[REASONING\].*',
        r'<\|.*?\|>',
    ]
    for pattern in cut_markers:
        cleaned = re.split(pattern, cleaned, flags=re.IGNORECASE | re.DOTALL)[0]

    # Gotcha/Gotchu repetition limiter (keep max 1 per response)
    gotcha_pattern = re.compile(r'\b[Gg]otch[aue]+\b')
    matches = list(gotcha_pattern.finditer(cleaned))
    if len(matches) > 1:
        alternatives = ["Got it", "Sure thing", "Right on", "Cool", "No problem"]
        for match in reversed(matches[1:]):
            replacement = random.choice(alternatives)
            cleaned = cleaned[:match.start()] + replacement + cleaned[match.end():]

    cleaned = cleaned.strip()
    if not cleaned or len(cleaned) < 10:
        cleaned = "The mists swirl with uncertainty... How may I assist you, seeker?"
    return cleaned


def create_llm_service() -> LLMService:
    """Factory: create appropriate LLM service based on settings."""
    settings = get_settings()
    if settings.LLM_MODE == "modal" and settings.MODAL_ENDPOINT:
        print(f"Using ModalLLMService: {settings.MODAL_ENDPOINT}")
        return ModalLLMService(settings.MODAL_ENDPOINT, settings.MODAL_API_KEY)
    print("Using MockLLMService")
    return MockLLMService()


print("clean_llm_response() and create_llm_service() defined")

---
# Part 6: Session Management

Player sessions are stored in a **Redis-backed session store** with 24-hour TTL.
If Redis is unavailable, sessions fall back to an in-memory dictionary.

Each session tracks:
- Archetype classification (from survey)
- Relationship score (evolves through interactions)
- Conversation history (last 10 turns)
- Last RL decision + context (for `/outcome` learning)
- Discount state (for adaptive discount escalation)

In [ ]:
# Cell 6.1: Player Session
# =========================

@dataclass
class PlayerSession:
    """Player session data persisted across interactions."""
    session_id: str
    archetype: PlayerArchetype
    participant_id: Optional[str] = None
    relationship_score: int = 50
    interaction_count: int = 0
    total_spent_points: int = 0
    conversation_history: List[Dict] = field(default_factory=list)
    wallet_address: Optional[str] = None
    last_decision: Optional[NPCDecision] = None
    last_context: Optional[GameContext] = None
    discount_offers_remaining: int = 1
    awaiting_discount_purchase: bool = False
    active_rl_discount: int = 0


print("PlayerSession defined")

In [ ]:
# Cell 6.2: Redis Session Store
# ===============================
#
# Provides async Redis-backed session persistence.
# Falls back to in-memory dict if Redis is unavailable.

import redis.asyncio as aioredis
import logging

logger = logging.getLogger(__name__)


class RedisSessionStore:
    """Redis-backed session storage with in-memory fallback."""

    def __init__(self, redis_url: Optional[str] = None):
        self.redis_url = redis_url
        self.redis_client: Optional[aioredis.Redis] = None
        self._local_cache: Dict[str, PlayerSession] = {}
        self._connected = False
        self.ttl = 86400  # 24 hours

    async def connect(self):
        if not self.redis_url:
            logger.warning("No REDIS_URL provided, using in-memory fallback")
            return False
        try:
            self.redis_client = aioredis.from_url(self.redis_url, encoding="utf-8", decode_responses=True)
            await self.redis_client.ping()
            self._connected = True
            logger.info("Redis session store connected")
            return True
        except Exception as e:
            logger.error(f"Redis connection failed: {e}")
            return False

    async def close(self):
        if self.redis_client:
            await self.redis_client.close()

    def _session_to_dict(self, session: PlayerSession) -> dict:
        d = {
            "session_id": session.session_id,
            "participant_id": getattr(session, 'participant_id', None),
            "archetype": session.archetype.value,
            "relationship_score": session.relationship_score,
            "interaction_count": session.interaction_count,
            "total_spent_points": session.total_spent_points,
            "conversation_history": session.conversation_history,
            "wallet_address": session.wallet_address,
            "discount_offers_remaining": session.discount_offers_remaining,
            "awaiting_discount_purchase": session.awaiting_discount_purchase,
            "active_rl_discount": session.active_rl_discount,
        }
        if session.last_decision:
            d["last_decision"] = session.last_decision.to_dict()
        if session.last_context:
            d["last_context"] = {
                "archetype": session.last_context.archetype.value,
                "relationship": session.last_context.relationship.value,
                "stock": session.last_context.stock.value,
                "risk": session.last_context.risk.value,
                "points": session.last_context.points.value,
                "has_debt": session.last_context.has_debt,
            }
        return d

    def _dict_to_session(self, data: dict) -> PlayerSession:
        session = PlayerSession(
            session_id=data["session_id"],
            archetype=PlayerArchetype(data["archetype"]),
            participant_id=data.get("participant_id"),
            relationship_score=data.get("relationship_score", 50),
            interaction_count=data.get("interaction_count", 0),
            total_spent_points=data.get("total_spent_points", 0),
            conversation_history=data.get("conversation_history", []),
            wallet_address=data.get("wallet_address"),
        )
        session.discount_offers_remaining = data.get("discount_offers_remaining", 1)
        session.awaiting_discount_purchase = data.get("awaiting_discount_purchase", False)
        session.active_rl_discount = data.get("active_rl_discount", 0)
        if "last_decision" in data and data["last_decision"]:
            ld = data["last_decision"]
            session.last_decision = NPCDecision(
                discount=DiscountAction(ld.get("discount_percent", 0)),
                loan=LoanAction(ld.get("loan_decision", "deny")),
                urgency=UrgencyAction(ld.get("urgency_level", "low")),
                upsell=UpsellAction(ld.get("upsell_recommendation", "none")),
                reasoning=ld.get("reasoning", "")
            )
        if "last_context" in data and data["last_context"]:
            lc = data["last_context"]
            session.last_context = GameContext(
                archetype=RLPlayerArchetype(lc["archetype"]),
                relationship=RelationshipLevel(lc["relationship"]),
                stock=StockLevel(lc["stock"]),
                risk=PlayerRisk(lc["risk"]),
                points=PointsLevel(lc["points"]),
                has_debt=lc.get("has_debt", False),
            )
        return session

    async def get(self, session_id: str) -> Optional[PlayerSession]:
        if not self._connected:
            return self._local_cache.get(session_id)
        try:
            data = await self.redis_client.get(f"session:{session_id}")
            if data:
                return self._dict_to_session(json.loads(data))
            return None
        except Exception as e:
            logger.error(f"Redis GET error: {e}")
            return self._local_cache.get(session_id)

    async def set(self, session_id: str, session: PlayerSession):
        self._local_cache[session_id] = session
        if not self._connected:
            return
        try:
            await self.redis_client.setex(f"session:{session_id}", self.ttl, json.dumps(self._session_to_dict(session)))
        except Exception as e:
            logger.error(f"Redis SET error: {e}")

    async def delete(self, session_id: str):
        self._local_cache.pop(session_id, None)
        if self._connected:
            try:
                await self.redis_client.delete(f"session:{session_id}")
            except Exception as e:
                logger.error(f"Redis DELETE error: {e}")

    async def exists(self, session_id: str) -> bool:
        if not self._connected:
            return session_id in self._local_cache
        try:
            return await self.redis_client.exists(f"session:{session_id}") > 0
        except:
            return session_id in self._local_cache

    async def get_all_sessions(self) -> Dict[str, PlayerSession]:
        if not self._connected:
            return self._local_cache.copy()
        try:
            sessions = {}
            cursor = 0
            while True:
                cursor, keys = await self.redis_client.scan(cursor, match="session:*", count=100)
                for key in keys:
                    data = await self.redis_client.get(key)
                    if data:
                        session = self._dict_to_session(json.loads(data))
                        sessions[session.session_id] = session
                if cursor == 0:
                    break
            return sessions
        except Exception as e:
            logger.error(f"Redis SCAN error: {e}")
            return self._local_cache.copy()

    async def count(self) -> int:
        if not self._connected:
            return len(self._local_cache)
        try:
            cursor, count = 0, 0
            while True:
                cursor, keys = await self.redis_client.scan(cursor, match="session:*", count=100)
                count += len(keys)
                if cursor == 0:
                    break
            return count
        except:
            return len(self._local_cache)

    @property
    def is_connected(self) -> bool:
        return self._connected


print("RedisSessionStore defined (async, with in-memory fallback)")

---
# Part 7: Helper Functions

Utility functions for archetype classification, context conversion, and game state helpers.

In [ ]:
# Cell 7.1: Helper Functions
# ===========================

import os

def game_state_to_context(game_state: GameState, archetype: str, relationship_score: int) -> GameContext:
    """Convert API GameState to RL GameContext."""
    stock_pct = ((game_state.hints_stock / max(1, game_state.hints_max)) +
                 (game_state.scrolls_stock / max(1, game_state.scrolls_max))) / 2 * 100
    return GameContext.from_raw(
        archetype=archetype, relationship_score=relationship_score,
        stock_percent=stock_pct, curses=game_state.curses,
        points=game_state.points, has_debt=game_state.has_loan,
    )


def get_relationship_level(score: int) -> str:
    if score <= 25: return "new"
    elif score <= 50: return "low"
    elif score <= 75: return "medium"
    return "high"


ARCHETYPE_DESCRIPTIONS = {
    PlayerArchetype.ACHIEVEMENT_HUNTER: "Driven by accomplishments and mastery.",
    PlayerArchetype.ENGAGED_BEGINNER: "Enthusiastic explorer of new experiences.",
    PlayerArchetype.SPENDER: "Values premium experiences and quality items.",
    PlayerArchetype.WEEKEND_WARRIOR: "Focused, extended gaming sessions.",
    PlayerArchetype.CASUAL_VETERAN: "Experienced, efficient player.",
    PlayerArchetype.TROPHY_HUNTER: "Achievement collector and skill prover."
}

STARTING_RESOURCES = {
    PlayerArchetype.SPENDER: (600, 50.0),
    PlayerArchetype.ACHIEVEMENT_HUNTER: (400, 20.0),
    PlayerArchetype.TROPHY_HUNTER: (350, 15.0),
    PlayerArchetype.WEEKEND_WARRIOR: (450, 25.0),
    PlayerArchetype.CASUAL_VETERAN: (500, 30.0),
    PlayerArchetype.ENGAGED_BEGINNER: (300, 10.0),
}


def classify_archetype(answers: SurveyAnswers) -> PlayerArchetype:
    """Map survey answers to player archetype."""
    if answers.spending == "regularly":
        return PlayerArchetype.SPENDER
    if answers.frequency == "rarely" and answers.session_length == "long":
        return PlayerArchetype.WEEKEND_WARRIOR
    if answers.frequency == "daily" and answers.session_length == "short":
        return PlayerArchetype.CASUAL_VETERAN
    if answers.motivation == "achievements" and answers.progression == "quick":
        return PlayerArchetype.TROPHY_HUNTER
    if answers.motivation == "achievements":
        return PlayerArchetype.ACHIEVEMENT_HUNTER
    return PlayerArchetype.ENGAGED_BEGINNER


print("Helper functions defined: classify_archetype, game_state_to_context, get_relationship_level")

---
# Part 8: FastAPI Application & Endpoints

This section defines:
1. **Lifespan** -- Startup/shutdown events (initialize LLM, RL agent, Redis, Stock Manager)
2. **Core Endpoints** -- `/health`, `/survey`, `/interact`, `/outcome`, `/session-end`
3. **RL Endpoints** -- `/rl/stats`, `/rl/prewarm`, `/rl/export`
4. **Admin Endpoints** -- Export, stats, stock management
5. **Proactive Endpoint** -- `/proactive` for context-aware commentary

In [ ]:
# Cell 8.1: Global State & Lifespan
# ===================================

from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from contextlib import asynccontextmanager

# Global state
sessions: Dict[str, PlayerSession] = {}
survey_responses: Dict[str, List[dict]] = {}
llm_service: LLMService = None
rl_agent: WhisperDecisionAgent = None
session_store: Optional[RedisSessionStore] = None


@asynccontextmanager
async def lifespan(app: FastAPI):
    global llm_service, rl_agent, session_store
    settings = get_settings()

    print()
    print("=" * 50)
    print("STARTING WHISPER NPC BACKEND")
    print("=" * 50)
    print(f"  Version: {settings.APP_VERSION}")
    print(f"  LLM Mode: {settings.LLM_MODE}")

    # Initialize Redis Session Store
    session_store = RedisSessionStore(settings.REDIS_URL)
    redis_ok = await session_store.connect()
    if redis_ok:
        print(f"  Redis Sessions: Connected")
    else:
        print(f"  Redis Sessions: Using in-memory fallback")

    # Initialize LLM
    llm_service = create_llm_service()

    # Initialize RL Agent
    rl_agent = WhisperDecisionAgent(
        epsilon=settings.RL_EPSILON,
        learning_rate=settings.RL_LEARNING_RATE,
    )
    print(f"  RL Agent: Loaded (epsilon={settings.RL_EPSILON}, lr={settings.RL_LEARNING_RATE})")

    print("=" * 50)
    print("Backend ready! Open http://127.0.0.1:8000/docs")
    print("=" * 50)
    print()

    yield

    # Cleanup
    print("Shutting down...")
    if session_store:
        await session_store.close()


# Create FastAPI app
app = FastAPI(
    title="Whisper NPC Backend",
    description="AI-powered NPC with RL decision-making for blockchain gaming",
    version="1.0.0",
    lifespan=lifespan
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

print("FastAPI app created with lifespan handler")

In [ ]:
# Cell 8.2: Core API Endpoints
# ==============================

@app.get("/health", response_model=HealthResponse)
async def health_check():
    settings = get_settings()
    llm_ok = await llm_service.health_check() if llm_service else False
    redis_ok = session_store.is_connected if session_store else False
    return HealthResponse(
        status="healthy" if llm_ok else "degraded",
        version=settings.APP_VERSION,
        llm_mode=settings.LLM_MODE,
        llm_loaded=llm_ok,
        rl_agent_loaded=rl_agent is not None,
        contexts_seen=len(rl_agent.seen_contexts) if rl_agent else 0,
        redis_sessions_connected=redis_ok,
    )


@app.post("/warmup")
async def warmup_modal():
    """Pre-warm Modal container before survey."""
    if not isinstance(llm_service, ModalLLMService):
        return {"status": "not_modal", "message": "Using local/mock LLM"}
    try:
        asyncio.create_task(llm_service.health_check())
        return {"status": "warming", "message": "Modal container starting"}
    except Exception as e:
        logger.warning(f"Warmup failed: {e}")


@app.post("/survey", response_model=SurveyResponse)
async def submit_survey(request: SurveyRequest):
    """Process player survey and assign archetype."""
    archetype = classify_archetype(request.answers)
    session = PlayerSession(
        session_id=request.session_id,
        archetype=archetype,
        participant_id=request.participant_id,
        wallet_address=request.wallet_address
    )
    await session_store.set(request.session_id, session)

    if request.participant_id:
        print(f"Session linked: {request.participant_id} -> {request.session_id}")

    points, pol = STARTING_RESOURCES.get(archetype, (400, 20.0))
    return SurveyResponse(
        session_id=request.session_id,
        archetype=archetype,
        archetype_description=ARCHETYPE_DESCRIPTIONS[archetype],
        initial_relationship=50,
        assigned_profile_id=f"profile_{archetype.value}_{request.session_id[:8]}",
        starting_points=points,
        starting_pol=pol,
    )


print("Core endpoints defined: /health, /warmup, /survey")

In [ ]:
# Cell 8.3: Main Interaction Endpoint (/interact)
# =================================================
#
# This is the primary NPC chat endpoint. Flow:
# 1. Build RL context from game state
# 2. Get RL decision (discount, urgency, upsell, loan)
# 3. Build LLM prompt with decision embedded
# 4. Generate dialogue via LLM
# 5. Clean response and update session

@app.post("/interact", response_model=NPCResponse)
async def interact_with_npc(request: InteractionRequest):
    session = await session_store.get(request.session_id)
    if not session:
        raise HTTPException(status_code=404, detail="Session not found. Complete /survey first.")

    if request.archetype_override:
        session.archetype = request.archetype_override

    start_time = time.time()
    settings = get_settings()

    # Build RL Context and get decision
    context = game_state_to_context(
        game_state=request.game_state,
        archetype=session.archetype.value,
        relationship_score=session.relationship_score,
    )
    decision = rl_agent.decide(context, explore=settings.DEBUG)

    # Store for /outcome learning
    session.last_decision = decision
    session.last_context = context

    # LLM sees Discount: 0% to prevent unsolicited deal-offering
    llm_decision = NPCDecision(
        discount=DiscountAction.NONE,
        urgency=decision.urgency,
        upsell=decision.upsell,
        loan=decision.loan,
        reasoning=decision.reasoning,
    )

    prompt = build_prompt(
        player_message=request.player_message,
        game_state=request.game_state.model_dump(),
        decision=llm_decision,
        archetype=session.archetype.value,
        relationship=get_relationship_level(session.relationship_score),
        conversation_history=session.conversation_history,
    )

    # Generate Response
    raw_dialogue, llm_latency = await llm_service.generate(prompt)
    dialogue = clean_llm_response(raw_dialogue)

    # Update Session
    session.interaction_count += 1
    relationship_delta = 2 if len(request.player_message) > 50 else 1
    session.relationship_score = min(100, session.relationship_score + relationship_delta)
    session.conversation_history.append({
        "player": request.player_message,
        "whisper": dialogue,
        "decision": decision.to_dict(),
        "timestamp": datetime.now().isoformat(),
    })

    # Feed relationship_delta to RL agent
    rl_agent.update_from_outcome(context, decision, {"relationship_delta": relationship_delta})

    total_latency = int((time.time() - start_time) * 1000)
    await session_store.set(request.session_id, session)

    decision_response = NPCDecisionResponse(
        discount_percent=decision.discount.value,
        loan_decision=decision.loan.value,
        urgency_level=decision.urgency.value,
        upsell_recommendation=decision.upsell.value,
        reasoning=decision.reasoning,
    ) if settings.DEBUG else None

    return NPCResponse(
        session_id=session.session_id,
        dialogue=dialogue,
        decision=decision_response,
        relationship_score=session.relationship_score,
        relationship_delta=relationship_delta,
        offer_discount_percent=decision.discount.value,
        latency_ms=total_latency,
    )


print("/interact endpoint defined")

In [ ]:
# Cell 8.4: Outcome & Session-End Endpoints
# ============================================

@app.post("/outcome")
async def report_outcome(request: OutcomeRequest):
    """Report player's decision for RL learning."""
    session = await session_store.get(request.session_id)
    if not session:
        raise HTTPException(status_code=404, detail="Session not found")

    if not session.last_decision or not session.last_context:
        if request.purchased and request.points_spent:
            session.total_spent_points += request.points_spent
            await session_store.set(request.session_id, session)
        return {"status": "skipped", "reason": "No prior RL decision to update"}

    outcome = {}
    if request.purchased is not None:
        outcome["purchased"] = 1.0 if request.purchased else 0.0
        if request.purchased and request.points_spent:
            session.total_spent_points += request.points_spent
    if request.upsell_accepted is not None:
        outcome["upsell_accepted"] = 1.0 if request.upsell_accepted else 0.0
    if request.loan_repaid is not None:
        outcome["loan_repaid"] = 1.0 if request.loan_repaid else 0.0
    if request.player_survived_level is not None:
        outcome["player_survived_level"] = 1.0 if request.player_survived_level else 0.0
    if request.session_continued is not None:
        outcome["session_continued"] = 1.0 if request.session_continued else 0.0
    if request.curses_after is not None:
        outcome["curses_after"] = float(request.curses_after)
    if request.final_points is not None:
        outcome["final_points"] = float(request.final_points)
    if request.relationship_score is not None:
        outcome["relationship_score"] = float(request.relationship_score)

    if outcome:
        rl_agent.update_from_outcome(session.last_context, session.last_decision, outcome)
        print(f"RL outcome recorded: {outcome}")

    if request.purchased and request.points_spent:
        await session_store.set(request.session_id, session)

    return {"status": "recorded", "outcome": outcome, "rl_stats": rl_agent.get_stats()}


@app.post("/session-end")
async def session_end(request: Request):
    """Handle session end -- applies quit penalty to RL."""
    try:
        data = await request.json()
        session_id = data.get("session_id")
        if not session_id:
            return {"status": "ignored", "reason": "no session_id"}
        session = await session_store.get(session_id)
        if session and session.last_context and session.last_decision:
            rl_agent.update_from_outcome(session.last_context, session.last_decision, {"player_quit": 1.0})
            print(f"Session ended (quit signal): {session_id}")
            return {"status": "recorded", "signal": "player_quit"}
        return {"status": "ignored", "reason": "no prior RL context"}
    except Exception as e:
        print(f"Session-end error: {e}")
        return {"status": "error", "message": str(e)}


@app.get("/session/{session_id}", response_model=SessionResponse)
async def get_session(session_id: str):
    session = await session_store.get(session_id)
    if not session:
        raise HTTPException(status_code=404, detail="Session not found")
    return SessionResponse(
        session_id=session.session_id, archetype=session.archetype,
        relationship_score=session.relationship_score,
        interaction_count=session.interaction_count,
        total_spent_points=session.total_spent_points,
    )


print("Endpoints defined: /outcome, /session-end, /session/{id}")

In [ ]:
# Cell 8.5: RL Management Endpoints
# ====================================

@app.get("/rl/stats")
async def get_rl_stats():
    if not rl_agent:
        raise HTTPException(status_code=500, detail="RL agent not initialized")
    return rl_agent.get_stats()


@app.post("/rl/prewarm")
async def prewarm_rl_agent(admin_key: str, n_iterations: int = 200):
    """Pre-warm the RL agent with simulated interactions from research hypotheses."""
    if admin_key != os.getenv("ADMIN_KEY", "admin2026"):
        raise HTTPException(status_code=403, detail="Unauthorized")
    if not rl_agent:
        raise HTTPException(status_code=500, detail="RL agent not initialized")

    current_stats = rl_agent.get_stats()
    if current_stats.get("contexts_seen", 0) > 50:
        return {"status": "already_prewarmed", "message": "Agent already has significant learning.", "current_stats": current_stats}

    prewarm_stats = rl_agent.prewarm(n_iterations)
    return {"status": "prewarmed", "iterations": n_iterations, "stats": prewarm_stats, "new_agent_stats": rl_agent.get_stats()}


@app.get("/rl/export")
async def export_rl_data(admin_key: str):
    """Export complete RL agent state for analysis."""
    if admin_key != os.getenv("ADMIN_KEY", "admin2026"):
        raise HTTPException(status_code=403, detail="Unauthorized")
    if not rl_agent:
        raise HTTPException(status_code=500, detail="RL agent not initialized")

    def serialize_q_values(bandit):
        return {ctx: {str(a.value): round(q, 4) for a, q in actions.items()} for ctx, actions in bandit.q_values.items()}

    return {
        "exported_at": datetime.now().isoformat(),
        "agent_config": {"epsilon": rl_agent.epsilon, "learning_rate": rl_agent.learning_rate},
        "summary_stats": rl_agent.get_stats(),
        "contexts_seen": list(rl_agent.seen_contexts),
        "bandits": {
            "discount": {"q_values": serialize_q_values(rl_agent.discount_bandit), "history_length": len(rl_agent.discount_bandit.history)},
            "urgency": {"q_values": serialize_q_values(rl_agent.urgency_bandit), "history_length": len(rl_agent.urgency_bandit.history)},
            "upsell": {"q_values": serialize_q_values(rl_agent.upsell_bandit), "history_length": len(rl_agent.upsell_bandit.history)},
            "loan": {"q_values": serialize_q_values(rl_agent.loan_bandit), "history_length": len(rl_agent.loan_bandit.history)},
        },
    }


print("RL endpoints defined: /rl/stats, /rl/prewarm, /rl/export")

In [ ]:
# Cell 8.6: Admin & Export Endpoints
# ====================================

@app.get("/admin/export/all")
async def export_all_data(admin_key: str):
    """Export all participant data as JSON for research analysis."""
    if admin_key != os.getenv("ADMIN_KEY", "admin2026"):
        raise HTTPException(status_code=403, detail="Unauthorized")

    export_data = {
        "exported_at": datetime.now().isoformat(),
        "participants": {},
        "survey_responses": survey_responses,
    }

    all_sessions = await session_store.get_all_sessions()
    for session_id, session in all_sessions.items():
        export_data["participants"][session_id] = {
            "session_id": session.session_id,
            "participant_id": getattr(session, 'participant_id', None),
            "archetype": session.archetype.value,
            "relationship_score": session.relationship_score,
            "interaction_count": session.interaction_count,
            "total_spent_points": session.total_spent_points,
            "conversation_history": session.conversation_history,
        }
    return export_data


@app.get("/admin/stats")
async def get_study_stats(admin_key: str):
    """Real-time study statistics for monitoring."""
    if admin_key != os.getenv("ADMIN_KEY", "admin2026"):
        raise HTTPException(status_code=403, detail="Unauthorized")

    all_sessions = await session_store.get_all_sessions()
    total = len(all_sessions)
    total_interactions = sum(s.interaction_count for s in all_sessions.values()) if all_sessions else 0
    avg_interactions = total_interactions / total if total > 0 else 0
    total_spent = sum(s.total_spent_points for s in all_sessions.values()) if all_sessions else 0
    avg_spent = total_spent / total if total > 0 else 0

    return {
        "total_participants": total,
        "avg_interactions_per_participant": round(avg_interactions, 1),
        "avg_points_spent": round(avg_spent, 1),
        "last_updated": datetime.now().isoformat(),
        "redis_connected": session_store.is_connected if session_store else False,
    }


print("Admin endpoints defined: /admin/export/all, /admin/stats")

---
# Part 9: Proactive Messaging

The proactive system generates **context-aware NPC commentary** triggered by game events
(puzzle solved, curse gained, golden gate found, etc.) without requiring the player to
initiate a conversation.

This makes the NPC feel **active** rather than purely reactive.

In [ ]:
# Cell 9.1: Proactive Messaging Endpoint
# =========================================

class ProactiveRequest(BaseModel):
    session_id: str
    trigger_type: str
    trigger_data: dict = {}
    game_state: dict


def build_trigger_context(trigger_type: str, trigger_data: dict, game_state: dict) -> str:
    """Build rich context string for each trigger type."""
    contexts = {
        "puzzle_solved": f"Player SOLVED the puzzle at Level {trigger_data.get('level', game_state.get('level', 1))}! Attempts: {trigger_data.get('attempts', 1)}. Congratulate them.",
        "golden_gate_found": f"Player found a GOLDEN GATE! Total: {trigger_data.get('total_golden', game_state.get('golden_gates_found', 0))}/5. Celebrate!",
        "curse_gained": f"Player gained a CURSE! Now at {trigger_data.get('current_curses', game_state.get('curses', 0))}/4. Express concern.",
        "near_game_over": f"URGENT: Player has 3/4 curses! Suggest a Gate Scroll if they can afford it (250 pts).",
        "level_complete": f"Player completed Level {trigger_data.get('completed_level', game_state.get('level', 1) - 1)}. Acknowledge progress.",
        "low_points": f"Player's points are low: {trigger_data.get('current_points', game_state.get('points', 0))}. Gently suggest POL conversion.",
        "game_start": "A new player entered the Gate of Whispers! Welcome them mysteriously.",
        "player_idle": "Player has been idle for 30+ seconds. Offer gentle guidance.",
    }
    return contexts.get(trigger_type, f"Something happened: {trigger_type}. React appropriately.")


@app.post("/proactive")
async def get_proactive_message(request: ProactiveRequest):
    """Generate context-aware proactive message using LLM."""
    start_time = time.time()
    session = await session_store.get(request.session_id)
    archetype = session.archetype if session else PlayerArchetype.ENGAGED_BEGINNER
    relationship_score = session.relationship_score if session else 50

    trigger_context = build_trigger_context(request.trigger_type, request.trigger_data, request.game_state)

    user_context = f"""[PROACTIVE EVENT - WHISPER NOTICES SOMETHING AND COMMENTS]
{trigger_context}

[CURRENT GAME STATE]
Level: {request.game_state.get('level', 1)}/7
Points: {request.game_state.get('points', 500)}
POL: {request.game_state.get('pol_balance', 75)}
Curses: {request.game_state.get('curses', 0)}/4
Golden Gates: {request.game_state.get('golden_gates_found', 0)}/5

[INSTRUCTIONS]
Generate a brief (1-2 sentences) contextual response as Whisper.
React specifically to the trigger event. Stay in character."""

    prompt = f"""<|im_start|>system
{WHISPER_SYSTEM_PROMPT}<|im_end|>
<|im_start|>user
{user_context}<|im_end|>
<|im_start|>assistant
"""

    try:
        raw_dialogue, llm_latency = await llm_service.generate(prompt)
        dialogue = clean_llm_response(raw_dialogue)
    except Exception as e:
        logger.warning(f"LLM generation failed: {e}")
        fallback = {
            'game_start': "Welcome, seeker. The Gate of Whispers awaits...",
            'curse_gained': "A curse marks your path... Be cautious.",
            'golden_gate_found': "A Golden Gate! Fortune smiles on you.",
            'level_complete': "Another level conquered. Well done.",
            'near_game_over': "Danger looms! Consider a Gate Scroll.",
        }
        dialogue = fallback.get(request.trigger_type, "The shadows stir...")

    # Relationship delta based on trigger type
    deltas = {"golden_gate_found": 3, "curse_gained": 2, "near_game_over": 2,
              "first_purchase": 5, "nft_purchased": 5, "puzzle_solved": 1, "level_complete": 2}
    relationship_delta = deltas.get(request.trigger_type, 1)

    if session:
        session.relationship_score = min(100, session.relationship_score + relationship_delta)

    total_latency = int((time.time() - start_time) * 1000)
    return {
        "dialogue": dialogue,
        "mood": request.trigger_type,
        "relationship_delta": relationship_delta,
        "latency_ms": total_latency,
        "trigger_type": request.trigger_type,
    }


print("/proactive endpoint defined")

---
# Part 10: Running the Server

## Option A: Run Locally
```bash
python main.py
# or
uvicorn main:app --reload --port 8000
```

## Option B: Run on Colab with ngrok
Use the cells below to start the server with a public URL.

In [ ]:
# Cell 10.1: Install and Configure ngrok (Colab only)
# =====================================================

!pip install -q pyngrok nest-asyncio

from pyngrok import ngrok
import nest_asyncio

nest_asyncio.apply()

# Set your ngrok auth token
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"

if NGROK_AUTH_TOKEN == "YOUR_NGROK_TOKEN_HERE":
    print("WARNING: Set your ngrok token above!")
    print("Get one free at: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok configured")

In [ ]:
# Cell 10.2: Start FastAPI Server with ngrok
# =============================================

import uvicorn
from threading import Thread
import requests

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

print("Starting FastAPI server...")
print("Waiting for server to be ready...")

# Wait for server
max_attempts = 30
for attempt in range(max_attempts):
    time.sleep(2)
    try:
        response = requests.get("http://localhost:8000/health", timeout=2)
        if response.status_code == 200:
            print(f"Server ready after {(attempt + 1) * 2} seconds")
            break
    except:
        if (attempt + 1) % 5 == 0:
            print(f"  [{(attempt + 1) * 2}s] Waiting for server...")

# Create ngrok tunnel
print()
print("Creating ngrok tunnel...")
tunnel = ngrok.connect(8000, bind_tls=True)
public_url = str(tunnel.public_url)

print()
print("=" * 60)
print("BACKEND RUNNING")
print("=" * 60)
print(f"  Public URL: {public_url}")
print(f"  API Docs:   {public_url}/docs")
print(f"  Health:     {public_url}/health")
print()
print("COPY THIS URL TO YOUR FRONTEND:")
print(f"  VITE_API_URL={public_url}")
print("=" * 60)

---
# Part 11: Test the Backend

In [ ]:
# Cell 11.1: Test the Backend
# ============================

import requests
import json

BASE_URL = public_url

print(f"Testing backend at: {BASE_URL}")
print()

# Test 1: Health check
print("1. Health Check...")
response = requests.get(f"{BASE_URL}/health")
print(f"   Status: {response.status_code}")
print(f"   Response: {response.json()}")
print()

# Test 2: Survey
print("2. Submit Survey...")
survey_data = {
    "session_id": "test_session_123",
    "answers": {
        "frequency": "daily",
        "session_length": "medium",
        "motivation": "achievements",
        "spending": "regularly",
        "progression": "both"
    }
}
response = requests.post(f"{BASE_URL}/survey", json=survey_data)
print(f"   Status: {response.status_code}")
survey_result = response.json()
print(f"   Archetype: {survey_result['archetype']}")
print(f"   Starting Points: {survey_result['starting_points']}")
print()

# Test 3: Interact with NPC
print("3. Interact with NPC...")
interact_data = {
    "session_id": "test_session_123",
    "player_message": "Hello! How much is a hint?",
    "game_state": {
        "level": 1,
        "points": 400,
        "pol_balance": 50.0,
        "curses": 0,
        "golden_gates_found": 0,
        "has_loan": False,
        "loan_amount": 0,
        "hints_stock": 100,
        "hints_max": 100,
        "scrolls_stock": 50,
        "scrolls_max": 100,
    }
}
response = requests.post(f"{BASE_URL}/interact", json=interact_data, timeout=120)
print(f"   Status: {response.status_code}")
npc_response = response.json()
print(f"   Whisper says: {npc_response['dialogue']}")
print(f"   Latency: {npc_response['latency_ms']}ms")
print(f"   Relationship: {npc_response['relationship_score']}")
print()

# Test 4: RL Stats
print("4. RL Agent Stats...")
response = requests.get(f"{BASE_URL}/rl/stats")
print(f"   {response.json()}")
print()

print("All tests passed!")
print(f"Backend ready at: {BASE_URL}")

---
# Summary

## Components Implemented

| Component | Description | Key Classes/Functions |
|-----------|-------------|----------------------|
| Configuration | Pydantic-based settings from .env | `Settings`, `get_settings()` |
| Enums | RL action/context spaces | `PlayerArchetype`, `DiscountAction`, `GameContext` |
| Request/Response | API schemas | `InteractionRequest`, `NPCResponse`, `GameState` |
| RL Agent | 4 contextual bandits + rule-based cold start | `WhisperDecisionAgent`, `ContextualBandit` |
| LLM Service | Modal (production) + Mock (testing) | `ModalLLMService`, `MockLLMService` |
| Sessions | Redis-backed with in-memory fallback | `RedisSessionStore`, `PlayerSession` |
| Prompt Builder | Qwen v2 ChatML format with RL decision | `build_prompt()` |
| Response Cleaner | Remove training artifacts | `clean_llm_response()` |
| Core Endpoints | Health, survey, interact, outcome | `/health`, `/survey`, `/interact` |
| RL Endpoints | Stats, pre-warm, export | `/rl/stats`, `/rl/prewarm` |
| Proactive | Context-aware NPC commentary | `/proactive` |
| Admin | Export, stats, stock management | `/admin/export/all` |

## Supporting Modules (imported in production main.py)

These modules are separate files in the production backend:

- **stock_manager.py** -- Global inventory with scarcity-driven dynamic pricing
- **action_executor.py** -- Intent detection, purchase confirmation, price negotiation
- **proactive_agent.py** -- Milestone detection and enhanced system prompts
- **auto_registration.py** -- Participant management with counterbalancing
- **modal_whisper.py** -- Modal.com LLM endpoint integration

## Running in Production

```bash
# Set environment variables
export LLM_MODE=modal
export MODAL_ENDPOINT=https://your-endpoint.modal.run/generate
export MODAL_API_KEY=your-key
export REDIS_URL=redis://your-redis:6379

# Start server
python main.py
```

---

**Notebook Version:** 5.0  
**Author:** Ramesh Krishnan  
**Date:** February 2026  
**Status:** Production